# 02 — Data Cleaning & Merging

**Goal:** Produce a single clean dataset ready for feature engineering.

**Steps:**
1. Build master team name mapping
2. Standardize team names in D1 and D2
3. Merge D1 + D2, deduplicate
4. Categorize tournament types (for ELO K-values)
5. Build confederation mapping
6. Filter D3 to Men's World Cup only
7. Output processed files

**Input:** `data/D1/`, `data/D2/`, `data/D3/`  
**Output:** `data/processed/matches_clean.csv`, `data/processed/team_confederations.csv`, `data/processed/tournament_categories.csv`

In [41]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


## Step 1 — Load Raw Data

In [42]:
d1 = pd.read_csv(DATA_DIR / "D1" / "results.csv", parse_dates=["date"])
d1_former = pd.read_csv(DATA_DIR / "D1" / "former_names.csv")

d2 = pd.read_csv(DATA_DIR / "D2" / "all_matches.csv", parse_dates=["date"])
d2_names = pd.read_csv(DATA_DIR / "D2" / "countries_names.csv")

d3_teams = pd.read_csv(DATA_DIR / "D3" / "teams.csv")
d3_matches = pd.read_csv(DATA_DIR / "D3" / "matches.csv")
d3_tournaments = pd.read_csv(DATA_DIR / "D3" / "tournaments.csv")

print(f"D1: {len(d1):,} rows")
print(f"D2: {len(d2):,} rows")
print(f"D3 matches: {len(d3_matches):,} rows")

D1: 49,215 rows
D2: 51,485 rows
D3 matches: 1,248 rows


## Step 2 — Build Master Team Name Mapping

We use D1's `former_names.csv` (historical → current) and D2's `countries_names.csv` (variants → canonical),
then add manual fixes for known cross-dataset mismatches (China PR, Czechia, etc.).

In [43]:
# Build name mapping: variant -> canonical
name_map = {}

# 1. From D1 former_names: historical name -> current name
for _, row in d1_former.iterrows():
    name_map[row['former']] = row['current']

# 2. From D2 countries_names: original -> current (only where they differ)
for _, row in d2_names.iterrows():
    if row['original_name'] != row['current_name']:
        name_map[row['original_name']] = row['current_name']

# 3. Manual cross-dataset fixes (found in exploration)
manual_fixes = {
    # D2 uses 'China', D1 uses 'China PR' (FIFA official name)
    'China': 'China PR',
    # D2 uses 'Czechia', D1/D3 use 'Czech Republic'
    'Czechia': 'Czech Republic',
    'Czech Republic': 'Czech Republic',  # prevent D2 remapping Czech Republic -> Czechia
    # D3 historical names not in D1 (D1 already modernized these)
    'West Germany': 'Germany',
    'Soviet Union': 'Russia',
    'Zaire': 'DR Congo',
    'East Germany': 'Germany',
    'Serbia and Montenegro': 'Serbia',
    'FR Yugoslavia': 'Serbia',
    'Yugoslavia': 'Serbia',
    'Dutch East Indies': 'Indonesia',
    'CIS': 'Russia',
    'Comm of Indep States': 'Russia',
    # Other known variants
    'Korea': 'South Korea',
    'Korea Republic': 'South Korea',
    "Côte d'Ivoire": 'Ivory Coast',
    'Cote d\'Ivoire': 'Ivory Coast',
    'IR Iran': 'Iran',
    'Trinidad & Tobago': 'Trinidad and Tobago',
    'Cape Verde Islands': 'Cape Verde',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'St. Kitts and Nevis': 'Saint Kitts and Nevis',
    'São Tomé and Príncipe': 'Sao Tome and Principe',
    'Timor-Leste': 'East Timor',
}
name_map.update(manual_fixes)

print(f"Total name mappings: {len(name_map)}")
print("\nSample mappings:")
for k, v in list(name_map.items())[:15]:
    print(f"  '{k}' -> '{v}'")

Total name mappings: 73

Sample mappings:
  'Dahomey' -> 'Benin'
  'Upper Volta' -> 'Burkina Faso'
  'Netherlands Antilles' -> 'Curaçao'
  'Bohemia' -> 'Czechia'
  'Bohemia and Moravia' -> 'Czechia'
  'Representation of Czechs and Slovaks' -> 'Czechoslovakia'
  'Belgian Congo' -> 'DR Congo'
  'Congo-Léopoldville' -> 'DR Congo'
  'Congo-Kinshasa' -> 'DR Congo'
  'Zaïre' -> 'DR Congo'
  'French Somaliland' -> 'Djibouti'
  'United Arab Republic' -> 'Egypt'
  'Swaziland' -> 'Eswatini'
  'Gold Coast' -> 'Ghana'
  'Portuguese Guinea' -> 'Guinea-Bissau'


In [44]:
def normalize_team_name(name, mapping):
    """Apply name mapping with one level of chaining.
    e.g. Bohemia -> Czechia -> Czech Republic resolves correctly."""
    result = mapping.get(name, name)
    return mapping.get(result, result)

# Apply to D1
d1['home_team'] = d1['home_team'].apply(lambda x: normalize_team_name(x, name_map))
d1['away_team'] = d1['away_team'].apply(lambda x: normalize_team_name(x, name_map))

# Apply to D2
d2['home_team'] = d2['home_team'].apply(lambda x: normalize_team_name(x, name_map))
d2['away_team'] = d2['away_team'].apply(lambda x: normalize_team_name(x, name_map))

print("Team names normalized in D1 and D2.")

# Verify key fixes
print(f"\n'China PR' in D1: {'China PR' in set(d1['home_team']) | set(d1['away_team'])}")
print(f"'China' in D1: {'China' in set(d1['home_team']) | set(d1['away_team'])}")
print(f"'China PR' in D2: {'China PR' in set(d2['home_team']) | set(d2['away_team'])}")
print(f"'Czech Republic' in D2: {'Czech Republic' in set(d2['home_team']) | set(d2['away_team'])}")
print(f"'Czechia' in D2: {'Czechia' in set(d2['home_team']) | set(d2['away_team'])}")

Team names normalized in D1 and D2.

'China PR' in D1: True
'China' in D1: False
'China PR' in D2: True
'Czech Republic' in D2: True
'Czechia' in D2: False


## Step 3 — Merge D1 + D2

Strategy:
- Merge on `(date, home_team, away_team, home_score, away_score)`
- When both datasets have the same match, prefer D1's row (has `city` column)
- When a match is only in D2, take D2's row (add empty `city` column)

In [45]:
# Add source tag before merge
d1['source'] = 'D1'
d2['source'] = 'D2'

# Add empty city column to D2
d2['city'] = None

# Align columns — use D1's column order
COLS = ['date', 'home_team', 'away_team', 'home_score', 'away_score',
        'tournament', 'city', 'country', 'neutral', 'source']
d1_aligned = d1[COLS].copy()
d2_aligned = d2[COLS].copy()

# Normalize neutral column type (D2 uses string 'True'/'False', D1 uses bool)
d1_aligned['neutral'] = d1_aligned['neutral'].astype(bool)
d2_aligned['neutral'] = d2_aligned['neutral'].apply(lambda x: str(x).strip().lower() == 'true')

# Concatenate D1 first, then D2
combined = pd.concat([d1_aligned, d2_aligned], ignore_index=True)
print(f"Combined before dedup: {len(combined):,}")

Combined before dedup: 100,700


In [46]:
# Deduplicate — keep D1 rows first (since D1 comes first in concat)
# Match key: (date, home_team, away_team, home_score, away_score)
MATCH_KEY = ['date', 'home_team', 'away_team', 'home_score', 'away_score']

combined_deduped = combined.drop_duplicates(subset=MATCH_KEY, keep='first').reset_index(drop=True)

print(f"After dedup: {len(combined_deduped):,}")
print(f"\nSource breakdown:")
print(combined_deduped['source'].value_counts())
print(f"\nDate range: {combined_deduped['date'].min().date()} to {combined_deduped['date'].max().date()}")

After dedup: 60,692

Source breakdown:
D1    49215
D2    11477
Name: source, dtype: int64

Date range: 1872-11-30 to 2026-03-31


In [47]:
# Sanity checks
print("Missing values after merge:")
print(combined_deduped.isnull().sum())
print(f"\nUnique teams: {len(set(combined_deduped['home_team']) | set(combined_deduped['away_team']))}")
print(f"Unique tournaments: {combined_deduped['tournament'].nunique()}")
print(f"\nSample rows from D2-only:")
combined_deduped[combined_deduped['source'] == 'D2'].head(3)

Missing values after merge:
date              0
home_team         0
away_team         0
home_score        0
away_score        0
tournament        0
city          11477
country           0
neutral           0
source            0
dtype: int64

Unique teams: 342
Unique tournaments: 338

Sample rows from D2-only:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,source
49215,1893-04-08,Northern Ireland,Wales,4,3,British Championship,None,Ireland,False,D2
49216,1902-04-05,Scotland,England,1,1,British Championship,None,Scotland,False,D2
49217,1902-08-09,Northern Ireland,Scotland,0,3,Friendly,None,Ireland,False,D2


## Step 4 — Tournament Categorization

Map all tournament names to 5 categories for ELO K-value assignment:
- `world_cup` (K=60)
- `continental_final` (K=50)
- `qualifier` (K=40)
- `other_competitive` (K=30)
- `friendly` (K=20)

In [48]:
# Rule-based categorization using keywords
def categorize_tournament(name):
    name_lower = str(name).lower()

    # Friendly (check first — some friendlies have 'cup' in name)
    if name_lower in ['friendly', 'friendly tournament', 'international friendly']:
        return 'friendly'

    # World Cup final tournament (not qualification)
    if 'world cup' in name_lower and 'qual' not in name_lower:
        return 'world_cup'

    # Qualifiers / Qualification rounds (includes 'qual' suffix like 'European Championship qual')
    if any(kw in name_lower for kw in ['qualif', 'qualifier', 'qualifying', 'olympic']) or name_lower.endswith(' qual'):
        return 'qualifier'

    # FIFA Series — warmup matches, treat as other_competitive
    if "fifa series" in name_lower:
        return "other_competitive"

    # Continental final tournaments (the actual tournaments, not qualifiers)
    continental_keywords = [
        'copa américa', 'copa america',
        'uefa euro', 'european championship',
        'african cup of nations', 'africa cup of nations', 'afcon',
        'afc asian cup', 'asian cup',
        'gold cup',
        'concacaf championship',
        'confederations cup',
        'nations league',
        'aff championship',
        'cosafa cup',
        'cecafa cup',
        'cfu caribbean cup',
        'waff championship',
        'saff cup',
        'eaff championship',
        'superclásico', 'superclasico',
        'conmebol',
        'oceania nations cup',
    ]
    if any(kw in name_lower for kw in continental_keywords) and 'qual' not in name_lower:
        return 'continental_final'

    # Everything else: other competitive
    return 'other_competitive'


combined_deduped['tournament_category'] = combined_deduped['tournament'].apply(categorize_tournament)

# ELO K-value mapping
K_VALUES = {
    'world_cup': 60,
    'continental_final': 50,
    'qualifier': 40,
    'other_competitive': 30,
    'friendly': 20,
}
combined_deduped['elo_k'] = combined_deduped['tournament_category'].map(K_VALUES)

print("Tournament category distribution:")
print(combined_deduped['tournament_category'].value_counts())
print(f"\nNull categories: {combined_deduped['tournament_category'].isnull().sum()}")

Tournament category distribution:
friendly             22075
qualifier            18676
other_competitive    10808
continental_final     7712
world_cup             1421
Name: tournament_category, dtype: int64

Null categories: 0


In [49]:
# Verify categorization on key tournaments
check_tournaments = [
    'FIFA World Cup', 'FIFA World Cup qualification', 'Friendly',
    'Copa América', 'UEFA Euro', 'African Cup of Nations',
    'AFC Asian Cup', 'Gold Cup', 'UEFA Nations League',
    'FIFA Series', 'Olympic Games', 'CECAFA Cup',
    'World Cup qualifier', 'European Championship qual',
    'African Nations Cup qualifier', 'Asian Cup qualifier',
]
print("Tournament categorization check:")
for t in check_tournaments:
    cat = categorize_tournament(t)
    print(f"  {t:45s} -> {cat} (K={K_VALUES[cat]})")

Tournament categorization check:
  FIFA World Cup                                -> world_cup (K=60)
  FIFA World Cup qualification                  -> qualifier (K=40)
  Friendly                                      -> friendly (K=20)
  Copa América                                  -> continental_final (K=50)
  UEFA Euro                                     -> continental_final (K=50)
  African Cup of Nations                        -> continental_final (K=50)
  AFC Asian Cup                                 -> continental_final (K=50)
  Gold Cup                                      -> continental_final (K=50)
  UEFA Nations League                           -> continental_final (K=50)
  FIFA Series                                   -> other_competitive (K=30)
  Olympic Games                                 -> qualifier (K=40)
  CECAFA Cup                                    -> continental_final (K=50)
  World Cup qualifier                           -> qualifier (K=40)
  European Champions

In [50]:
# Review other_competitive — check for any misclassified important tournaments
other = combined_deduped[combined_deduped['tournament_category'] == 'other_competitive']
print(f"other_competitive matches: {len(other):,}")
print(f"\nTop 30 tournaments in other_competitive:")
print(other['tournament'].value_counts().head(30))

other_competitive matches: 10,808

Top 30 tournaments in other_competitive:
Merdeka Tournament                    599
Gulf Cup                              577
Asian Games                           525
British Home Championship             523
Island Games                          395
King's Cup                            338
South Pacific Games                   298
Nordic Championship                   290
African Nations Cup                   278
Arab Cup                              250
Independence Tournament               242
Amílcar Cabral Cup                    235
Southeast Asian Games                 221
Muratti Vase                          220
CCCF Championship                     183
UNCAF Cup                             164
Korea Cup                             159
Pan Arab Games                        139
Central European International Cup    126
Baltic Cup                            120
Indian Ocean Island Games             110
C Am & Caribbean Games                104


In [51]:
# If any tournaments are misclassified above, add them to categorize_tournament() and re-run
# Then save the tournament category reference file
tournament_ref = (
    combined_deduped[['tournament', 'tournament_category', 'elo_k']]
    .drop_duplicates(subset='tournament')
    .sort_values('tournament_category')
    .reset_index(drop=True)
)
tournament_ref.to_csv(PROCESSED_DIR / 'tournament_categories.csv', index=False)
print(f"Saved tournament_categories.csv — {len(tournament_ref)} unique tournaments categorized")

Saved tournament_categories.csv — 338 unique tournaments categorized


## Step 5 — Confederation Mapping

D3 gives us confederation data for 85 teams (those that ever appeared in a World Cup).
We supplement this for all other teams using a manual mapping.

In [52]:
# Extract confederation data from D3 (Men's teams only)
d3_conf = d3_teams[d3_teams['mens_team'] == 1][['team_name', 'confederation_code', 'confederation_name']].copy()

# Apply same name normalization to D3 team names
d3_conf['team_name'] = d3_conf['team_name'].apply(lambda x: normalize_team_name(x, name_map))

print(f"D3 confederation data: {len(d3_conf)} teams")
print(d3_conf['confederation_code'].value_counts())

D3 confederation data: 85 teams
UEFA        39
CAF         13
AFC         12
CONCACAF    11
CONMEBOL     9
OFC          1
Name: confederation_code, dtype: int64


In [53]:
# Find teams in our merged dataset that don't have confederation data from D3
all_teams = set(combined_deduped['home_team']) | set(combined_deduped['away_team'])
d3_conf_teams = set(d3_conf['team_name'])
missing_conf = sorted(all_teams - d3_conf_teams)
print(f"Teams missing confederation data: {len(missing_conf)}")
print("\nMissing teams:")
for t in missing_conf:
    print(f"  - {t}")

Teams missing confederation data: 263

Missing teams:
  - Abkhazia
  - Afghanistan
  - Albania
  - Alderney
  - Ambazonia
  - American Samoa
  - Andalusia
  - Andorra
  - Anguilla
  - Antigua and Barbuda
  - Arameans Suryoye
  - Armenia
  - Artsakh
  - Aruba
  - Asturias
  - Aymara
  - Azerbaijan
  - Bahamas
  - Bahrain
  - Bangladesh
  - Barawa
  - Barbados
  - Basque Country
  - Belarus
  - Belize
  - Benin
  - Bermuda
  - Bhutan
  - Biafra
  - Bonaire
  - Botswana
  - British Virgin Islands
  - Brittany
  - Brunei
  - Burkina Faso
  - Burundi
  - Cambodia
  - Canary Islands
  - Cape Verde
  - Cascadia
  - Catalonia
  - Cayman Islands
  - Central African Republic
  - Central Spain
  - Chad
  - Chagos Islands
  - Chameria
  - Chechnya
  - Cilento
  - Comoros
  - Congo
  - Cook Islands
  - Corsica
  - County of Nice
  - Crimea
  - Curaçao
  - Cyprus
  - Darfur
  - Djibouti
  - Dominica
  - Dominican Republic
  - Donetsk PR
  - Délvidék
  - East Timor
  - Elba Island
  - Ellan Vannin
  

In [54]:
# Manual confederation assignments for teams not in D3
# (FIFA member nations + CONIFA teams that appear in our dataset)
manual_conf = {
    # UEFA
    'Albania': 'UEFA', 'Andorra': 'UEFA', 'Armenia': 'UEFA', 'Azerbaijan': 'UEFA', 'Belarus': 'UEFA',
    'Cyprus': 'UEFA', 'Finland': 'UEFA', 'Monaco': 'UEFA',
    'Estonia': 'UEFA', 'Faroe Islands': 'UEFA', 'Georgia': 'UEFA', 'Gibraltar': 'UEFA',
    'Kosovo': 'UEFA', 'Latvia': 'UEFA', 'Liechtenstein': 'UEFA', 'Lithuania': 'UEFA',
    'Luxembourg': 'UEFA', 'Malta': 'UEFA', 'Moldova': 'UEFA', 'Montenegro': 'UEFA',
    'North Macedonia': 'UEFA', 'San Marino': 'UEFA', 'Slovakia': 'UEFA',
    'Slovenia': 'UEFA', 'Ukraine': 'UEFA', 'Northern Ireland': 'UEFA',
    'Scotland': 'UEFA', 'Wales': 'UEFA', 'England': 'UEFA', 'Republic of Ireland': 'UEFA',
    'Czechia': 'UEFA', 'Czech Republic': 'UEFA', 'Kosovo': 'UEFA',
    # CAF
    'Benin': 'CAF', 'Botswana': 'CAF', 'Burkina Faso': 'CAF', 'Burundi': 'CAF',
    'Cape Verde': 'CAF', 'Central African Republic': 'CAF', 'Chad': 'CAF',
    'Comoros': 'CAF', 'Congo': 'CAF', 'Djibouti': 'CAF', 'Equatorial Guinea': 'CAF',
    'Eritrea': 'CAF', 'Eswatini': 'CAF', 'Ethiopia': 'CAF', 'Gabon': 'CAF',
    'Gambia': 'CAF', 'Guinea': 'CAF', 'Guinea-Bissau': 'CAF', 'Kenya': 'CAF',
    'Lesotho': 'CAF', 'Liberia': 'CAF', 'Libya': 'CAF', 'Madagascar': 'CAF',
    'Malawi': 'CAF', 'Mali': 'CAF', 'Mauritania': 'CAF', 'Mauritius': 'CAF',
    'Mozambique': 'CAF', 'Namibia': 'CAF', 'Niger': 'CAF', 'Rwanda': 'CAF',
    'Sao Tome and Principe': 'CAF', 'Sierra Leone': 'CAF', 'Somalia': 'CAF',
    'South Sudan': 'CAF', 'Sudan': 'CAF', 'Tanzania': 'CAF', 'Togo': 'CAF',
    'Uganda': 'CAF', 'Zambia': 'CAF', 'Zimbabwe': 'CAF',
    'Seychelles': 'CAF', 'South Africa': 'CAF',
    # AFC
    'Afghanistan': 'AFC', 'Bahrain': 'AFC', 'Bangladesh': 'AFC', 'Bhutan': 'AFC',
    'Brunei': 'AFC', 'Cambodia': 'AFC', 'China PR': 'AFC', 'Guam': 'AFC',
    'Hong Kong': 'AFC', 'India': 'AFC', 'Indonesia': 'AFC', 'Jordan': 'AFC',
    'Kazakhstan': 'AFC', 'Kuwait': 'AFC', 'Kyrgyzstan': 'AFC', 'Laos': 'AFC',
    'Lebanon': 'AFC', 'Macau': 'AFC', 'Maldives': 'AFC', 'Mongolia': 'AFC',
    'Myanmar': 'AFC', 'Nepal': 'AFC', 'North Korea': 'AFC', 'Oman': 'AFC',
    'Pakistan': 'AFC', 'Palestine': 'AFC', 'Philippines': 'AFC', 'Singapore': 'AFC',
    'South Korea': 'AFC', 'Sri Lanka': 'AFC', 'Syria': 'AFC', 'Taiwan': 'AFC',
    'Tajikistan': 'AFC', 'Thailand': 'AFC', 'Timor-Leste': 'AFC', 'East Timor': 'AFC',
    'Turkmenistan': 'AFC', 'Uzbekistan': 'AFC', 'Vietnam': 'AFC', 'Yemen': 'AFC',
    'Malaysia': 'AFC', 'Australia': 'AFC',
    # CONCACAF
    'Anguilla': 'CONCACAF', 'Antigua and Barbuda': 'CONCACAF', 'Aruba': 'CONCACAF',
    'Bahamas': 'CONCACAF', 'Barbados': 'CONCACAF', 'Belize': 'CONCACAF',
    'Bermuda': 'CONCACAF', 'British Virgin Islands': 'CONCACAF', 'Cayman Islands': 'CONCACAF',
    'Cuba': 'CONCACAF', 'Curaçao': 'CONCACAF', 'Dominica': 'CONCACAF',
    'Dominican Republic': 'CONCACAF', 'El Salvador': 'CONCACAF', 'French Guiana': 'CONCACAF',
    'Grenada': 'CONCACAF', 'Guadeloupe': 'CONCACAF', 'Guatemala': 'CONCACAF',
    'Guyana': 'CONCACAF', 'Haiti': 'CONCACAF', 'Honduras': 'CONCACAF',
    'Jamaica': 'CONCACAF', 'Martinique': 'CONCACAF', 'Montserrat': 'CONCACAF',
    'Nicaragua': 'CONCACAF', 'Panama': 'CONCACAF', 'Puerto Rico': 'CONCACAF',
    'Saint Kitts and Nevis': 'CONCACAF', 'Saint Lucia': 'CONCACAF',
    'St Vincent & Grenadines': 'CONCACAF', 'Suriname': 'CONCACAF',
    'Trinidad and Tobago': 'CONCACAF', 'Turks and Caicos Islands': 'CONCACAF',
    'US Virgin Islands': 'CONCACAF',
    # CONMEBOL
    'Colombia': 'CONMEBOL', 'Ecuador': 'CONMEBOL', 'Venezuela': 'CONMEBOL',
    # OFC
    'Cook Islands': 'OFC', 'Fiji': 'OFC', 'New Caledonia': 'OFC',
    'New Zealand': 'OFC', 'Papua New Guinea': 'OFC', 'Samoa': 'OFC',
    'Solomon Islands': 'OFC', 'Tahiti': 'OFC', 'Tonga': 'OFC', 'Vanuatu': 'OFC',
    'American Samoa': 'OFC', 'Niue': 'OFC', 'Kiribati': 'OFC',
    'Marshall Islands': 'OFC', 'Palau': 'OFC', 'Tuvalu': 'OFC',
    # Unknown / CONIFA (non-FIFA) — mark as UNKNOWN, won't appear in WC predictions
}

# Build final confederation DataFrame
d3_conf_dict = dict(zip(d3_conf['team_name'], d3_conf['confederation_code']))
all_conf = {**manual_conf, **d3_conf_dict}  # D3 data overrides manual (more authoritative)

conf_df = pd.DataFrame([
    {'team': team, 'confederation': all_conf.get(team, 'UNKNOWN')}
    for team in sorted(all_teams)
])

print(f"Confederation mapping: {len(conf_df)} teams")
print(f"\nBreakdown:")
print(conf_df['confederation'].value_counts())

Confederation mapping: 342 teams

Breakdown:
UNKNOWN     122
UEFA         55
CAF          54
AFC          47
CONCACAF     38
OFC          16
CONMEBOL     10
Name: confederation, dtype: int64


In [55]:
# Check which teams still have UNKNOWN confederation
unknown = conf_df[conf_df['confederation'] == 'UNKNOWN']
print(f"Teams with UNKNOWN confederation: {len(unknown)}")
if len(unknown) > 0:
    print("\nThese teams will be excluded from 2026 predictions (non-FIFA entities):")
    for t in unknown['team'].tolist():
        print(f"  - {t}")

Teams with UNKNOWN confederation: 122

These teams will be excluded from 2026 predictions (non-FIFA entities):
  - Abkhazia
  - Alderney
  - Ambazonia
  - Andalusia
  - Arameans Suryoye
  - Artsakh
  - Asturias
  - Aymara
  - Barawa
  - Basque Country
  - Biafra
  - Bonaire
  - Brittany
  - Canary Islands
  - Cascadia
  - Catalonia
  - Central Spain
  - Chagos Islands
  - Chameria
  - Chechnya
  - Cilento
  - Corsica
  - County of Nice
  - Crimea
  - Darfur
  - Donetsk PR
  - Délvidék
  - Elba Island
  - Ellan Vannin
  - FS Micronesia
  - Falkland Islands
  - Felvidék
  - Franconia
  - Frøya
  - Galicia
  - German DR
  - Gotland
  - Gozo
  - Greenland
  - Guernsey
  - Găgăuzia
  - Hitra
  - Hmong
  - Iraqi Kurdistan
  - Isle of Man
  - Isle of Wight
  - Jersey
  - Kabylia
  - Kernow
  - Kurdistan
  - Kárpátalja
  - Luhansk PR
  - Macao
  - Madrid
  - Manchukuo
  - Mapuche
  - Matabeleland
  - Maule Sur
  - Mayotte
  - Menorca
  - Micronesia
  - North Vietnam
  - North Yemen
  - Norther

In [56]:
# Save confederation mapping
conf_df.to_csv(PROCESSED_DIR / 'team_confederations.csv', index=False)
print(f"Saved team_confederations.csv — {len(conf_df)} teams")

Saved team_confederations.csv — 342 teams


## Step 6 — Add Confederation to Matches & Final Cleanup

In [57]:
# Add confederation columns to matches
conf_lookup = dict(zip(conf_df['team'], conf_df['confederation']))

combined_deduped['home_confederation'] = combined_deduped['home_team'].map(conf_lookup).fillna('UNKNOWN')
combined_deduped['away_confederation'] = combined_deduped['away_team'].map(conf_lookup).fillna('UNKNOWN')

# Add match outcome column (target variable)
def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

combined_deduped['outcome'] = combined_deduped.apply(get_outcome, axis=1)
combined_deduped['goal_diff'] = combined_deduped['home_score'] - combined_deduped['away_score']

# Sort chronologically
combined_deduped = combined_deduped.sort_values('date').reset_index(drop=True)

print("Outcome distribution:")
print(combined_deduped['outcome'].value_counts())
print(f"\nFinal columns: {list(combined_deduped.columns)}")

Outcome distribution:
home_win    31836
away_win    15262
draw        13594
Name: outcome, dtype: int64

Final columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'source', 'tournament_category', 'elo_k', 'home_confederation', 'away_confederation', 'outcome', 'goal_diff']


## Step 7 — Filter D3 to Men's World Cup

In [58]:
# Get men's tournament IDs from D3
mens_tournaments = d3_tournaments[d3_tournaments['tournament_name'].str.contains("Men's")]['tournament_id'].tolist()
print(f"Men's World Cup tournament IDs: {mens_tournaments}")

d3_mens = d3_matches[d3_matches['tournament_id'].isin(mens_tournaments)].copy()

# Normalize team names in D3
d3_mens['home_team_name'] = d3_mens['home_team_name'].apply(lambda x: normalize_team_name(x, name_map))
d3_mens['away_team_name'] = d3_mens['away_team_name'].apply(lambda x: normalize_team_name(x, name_map))

print(f"D3 Men's WC matches: {len(d3_mens)}")
print(f"\nTournaments:")
print(d3_mens['tournament_name'].value_counts())

Men's World Cup tournament IDs: ['WC-1930', 'WC-1934', 'WC-1938', 'WC-1950', 'WC-1954', 'WC-1958', 'WC-1962', 'WC-1966', 'WC-1970', 'WC-1974', 'WC-1978', 'WC-1982', 'WC-1986', 'WC-1990', 'WC-1994', 'WC-1998', 'WC-2002', 'WC-2006', 'WC-2010', 'WC-2014', 'WC-2018', 'WC-2022']
D3 Men's WC matches: 964

Tournaments:
2022 FIFA Men's World Cup    64
2018 FIFA Men's World Cup    64
2014 FIFA Men's World Cup    64
2010 FIFA Men's World Cup    64
2006 FIFA Men's World Cup    64
2002 FIFA Men's World Cup    64
1998 FIFA Men's World Cup    64
1986 FIFA Men's World Cup    52
1994 FIFA Men's World Cup    52
1990 FIFA Men's World Cup    52
1982 FIFA Men's World Cup    52
1978 FIFA Men's World Cup    38
1974 FIFA Men's World Cup    38
1958 FIFA Men's World Cup    35
1970 FIFA Men's World Cup    32
1966 FIFA Men's World Cup    32
1962 FIFA Men's World Cup    32
1954 FIFA Men's World Cup    26
1950 FIFA Men's World Cup    22
1938 FIFA Men's World Cup    18
1930 FIFA Men's World Cup    18
1934 FIFA Men'

## Step 8 — Save Outputs

In [59]:
# Save main cleaned dataset
combined_deduped.to_csv(PROCESSED_DIR / 'matches_clean.csv', index=False)
print(f"Saved matches_clean.csv — {len(combined_deduped):,} rows")

# Save D3 Men's WC data
d3_mens.to_csv(PROCESSED_DIR / 'd3_mens_wc.csv', index=False)
print(f"Saved d3_mens_wc.csv — {len(d3_mens):,} rows")

# Save name mapping for reference
name_map_df = pd.DataFrame([
    {'variant': k, 'canonical': v} for k, v in sorted(name_map.items())
])
name_map_df.to_csv(PROCESSED_DIR / 'team_name_mapping.csv', index=False)
print(f"Saved team_name_mapping.csv — {len(name_map_df)} mappings")

Saved matches_clean.csv — 60,692 rows
Saved d3_mens_wc.csv — 964 rows
Saved team_name_mapping.csv — 73 mappings


In [60]:
# Final summary
print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"\nInput:")
print(f"  D1: 49,215 matches")
print(f"  D2: 51,485 matches")
print(f"  D3: 1,248 WC matches")
print(f"\nOutput (matches_clean.csv):")
print(f"  Total matches: {len(combined_deduped):,}")
print(f"  Date range: {combined_deduped['date'].min().date()} to {combined_deduped['date'].max().date()}")
print(f"  Unique teams: {len(set(combined_deduped['home_team']) | set(combined_deduped['away_team']))}")
print(f"  Tournament categories: {combined_deduped['tournament_category'].value_counts().to_dict()}")
print(f"  Outcome split: {combined_deduped['outcome'].value_counts().to_dict()}")
print(f"  Missing values: {combined_deduped.isnull().sum().sum()} total")
print(f"\nReference files:")
print(f"  team_confederations.csv")
print(f"  tournament_categories.csv")
print(f"  team_name_mapping.csv")
print(f"  d3_mens_wc.csv")
print(f"\nNext: 03_feature_engineering.ipynb")

DATA CLEANING SUMMARY

Input:
  D1: 49,215 matches
  D2: 51,485 matches
  D3: 1,248 WC matches

Output (matches_clean.csv):
  Total matches: 60,692
  Date range: 1872-11-30 to 2026-03-31
  Unique teams: 342
  Tournament categories: {'friendly': 22075, 'qualifier': 18676, 'other_competitive': 10808, 'continental_final': 7712, 'world_cup': 1421}
  Outcome split: {'home_win': 31836, 'away_win': 15262, 'draw': 13594}
  Missing values: 11477 total

Reference files:
  team_confederations.csv
  tournament_categories.csv
  team_name_mapping.csv
  d3_mens_wc.csv

Next: 03_feature_engineering.ipynb
